# 13. Speech Translation — Single-Shot Speech-to-Text Translation from Microphone (Speech SDK)

This notebook uses the Azure AI Speech SDK's **`TranslationRecognizer`** — a recognizer that combines speech
recognition and text translation in one call graph: it listens to the microphone, recognizes the spoken
English, and translates the recognized text into French, all in a single `recognize_once_async()` call.

**⚠️ Requires a live microphone.** Unlike notebook 11's *continuous* recognition (which can hang indefinitely
without a stop event), this notebook uses **single-shot** recognition (`recognize_once_async()`), so it will
return after one utterance or a timeout — but it still needs a working microphone and something spoken into it
to produce a real result. If no microphone is available, **run `13_speech_translate.py` as a standalone script
from a terminal** on a machine with a mic instead of executing the cells below.

**Difficulty:** Intermediate


## Prerequisites

**Python packages:**
- `azure-cognitiveservices-speech` — **not yet in the repo's root `requirements.txt`**, install with:
  ```
  pip install azure-cognitiveservices-speech
  ```
- `python-dotenv` (already in `requirements.txt`)

**Azure resources required:**
- An Azure AI Speech resource with a key and its **region** (this script uses region-based config, not an
  endpoint URL, unlike notebooks 10–12)

**Hardware required:**
- A local microphone / audio input device

**Environment variables** (add to root `.env`), following the pattern requested for region-based Speech
config:
```
AZURE_SPEECH_KEY=<your-speech-resource-key>
AZURE_SPEECH_REGION=<your-speech-resource-region>
```


## What You'll Learn

- How `SpeechTranslationConfig` differs from plain `SpeechConfig` (adds recognition + translation language
  config)
- How to set the spoken source language (`speech_recognition_language`) and add one or more target languages
  (`add_target_language`)
- How `TranslationRecognizer` combines recognition and translation into one operation
- How single-shot recognition (`recognize_once_async`) differs from continuous recognition (notebook 11)


### Step 1 — Configure speech translation: source language and target(s)

`SpeechTranslationConfig(subscription=..., region=...)` is the translation-specific counterpart to
`SpeechConfig` — notice it's constructed with `region` (a short string like `"eastus2"`) rather than a full
`endpoint` URL, the other valid way to point the SDK at your resource (see notebook 11's `endpoint=` form).

`speech_recognition_language = "en-US"` tells the recognizer what language is being *spoken*.
`add_target_language("fr")` adds French as an output translation language — this can be called multiple times
to translate into several languages from one recognized utterance.

💡 **Exam tip:** `SpeechTranslationConfig` and `TranslationRecognizer` are a distinct object pair from
`SpeechConfig`/`SpeechRecognizer` (notebook 11) — AI-102 expects you to know that speech translation is its
own recognizer type built specifically to combine recognition + translation, rather than something you bolt
onto a plain `SpeechRecognizer`.

🔄 **Alternatives:** Call `add_target_language()` more than once (e.g. also `"ja"`, `"es"`) to get translations
into multiple languages from a single recognized utterance, analogous to notebook 09's multi-target Translator
REST call but for speech instead of text.


In [ ]:
import os
from dotenv import load_dotenv
import azure.cognitiveservices.speech as speechsdk

load_dotenv()

api_key = os.environ["AZURE_SPEECH_KEY"]
region = os.environ["AZURE_SPEECH_REGION"]

translation_config = speechsdk.translation.SpeechTranslationConfig(
    subscription=api_key,
    region=region
)

translation_config.speech_recognition_language = "en-US"
translation_config.add_target_language("fr")

### Step 2 — Create the recognizer and listen for one utterance

`AudioConfig(use_default_microphone=True)` captures from the default mic (same as notebook 11).
`TranslationRecognizer(translation_config, audio_config)` is the combined recognition+translation object.
`recognize_once_async().get()` performs **single-shot** recognition: it listens for one utterance (until a
pause/silence is detected or a timeout is hit), recognizes it, translates it, and returns — no event handlers
or polling loop required, unlike notebook 11's continuous recognition.

💡 **Exam tip:** `recognize_once_async()` is the right choice for short, discrete commands/utterances (voice
commands, short Q&A turns); continuous recognition (`start_continuous_recognition()` + events, notebook 11) is
the right choice for open-ended dictation or ongoing conversation capture. Know when the exam scenario implies
one vs. the other.

🔄 **Alternatives:** For a longer conversation you want translated as it happens (not just one utterance), use
**continuous translation recognition** — `TranslationRecognizer` also supports
`start_continuous_recognition()` with `.recognized`/`.recognizing` event handlers, mirroring notebook 11's
event-driven pattern but with translated output as well.


In [ ]:
audio_config = speechsdk.audio.AudioConfig(use_default_microphone=True)

translation_recognizer = speechsdk.translation.TranslationRecognizer(
    translation_config=translation_config,
    audio_config=audio_config
)

print("Listening...")

# Requires a live microphone. If none is available, run 13_speech_translate.py
# as a standalone script instead of executing this cell.
result = translation_recognizer.recognize_once_async().get()

### Step 3 — Print the recognized text and its translation

`result.reason == speechsdk.ResultReason.TranslatedSpeech` confirms both recognition and translation
succeeded. `result.text` is the recognized source-language text; `result.translations` is a dict keyed by
target language code (here, `"fr"`) mapping to the translated string — this dict would have one entry per
language you called `add_target_language()` for.

💡 **Exam tip:** `ResultReason.TranslatedSpeech` is a distinct enum value from plain
`ResultReason.RecognizedSpeech` (used by `SpeechRecognizer` in notebook 11) — `TranslationRecognizer` results
use the translation-specific reason to confirm the full recognize-then-translate pipeline completed, not just
recognition alone.

🔄 **Alternatives:** Set `translation_config.voice_name` to a target-language neural voice name to also get
**synthesized spoken audio** of the translation (speech-to-speech translation), not just translated text —
handled via the recognizer's `.synthesizing` event, combining this notebook's translation with notebook 12's
synthesis technique.


In [ ]:
if result.reason == speechsdk.ResultReason.TranslatedSpeech:
    print("\nOriginal text:")
    print(result.text)

    print("\nFrench translation:")
    print(result.translations["fr"])

## Summary

This notebook used `TranslationRecognizer` to listen for one spoken English utterance and return both the
recognized text and its French translation in a single `recognize_once_async()` call. Compared to chaining
separate speech-to-text (notebook 10/11) and text-translation (notebook 04/09) steps, a dedicated translation
recognizer handles both in one integrated pipeline — and can be extended to synthesize spoken translated output
too.


## Try It Yourself

1. **Easy:** Add a second target language with `translation_config.add_target_language("ja")` and print
   `result.translations["ja"]` alongside the French output.
2. **Intermediate:** Swap `recognize_once_async()` for continuous translation recognition
   (`start_continuous_recognition()` with `.recognized`/`.recognizing` handlers, following notebook 11's event
   pattern) so a longer conversation can be translated turn by turn.
3. **Advanced:** Set `translation_config.voice_name` to a French neural voice and handle the `.synthesizing`
   event to save the spoken French translation to a WAV file, combining this notebook's translation with
   notebook 12's text-to-speech technique for full speech-to-speech translation.
